In [1]:
# African Economic Crisis Analysis
# Analisis Krisis Ekonomi Benua Afrika

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_regression, RFE
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("Analisis Krisis Ekonomi Benua Afrika - Dimulai")

Libraries imported successfully!
Analisis Krisis Ekonomi Benua Afrika - Dimulai


In [3]:
df = pd.read_csv('african_crises.csv')
df.head()

,Country,year,systemic_crisis,exch_usd,domestic_debt_in_default,sovereign_external_debt_default,gdp_weighted_default,inflation_annual_cpi,independence,currency_crises,inflation_crises,banking_crisis
0,Zimbabwe,1870,1,0.052264,0,0,0.0,3.441456,0,0,0,crisis
1,Zimbabwe,1871,0,0.052798,0,0,0.0,14.149140,0,0,0,no_crisis
2,Zimbabwe,1872,0,0.052274,0,0,0.0,-3.718593,0,0,0,no_crisis
3,Zimbabwe,1873,0,0.051680,0,0,0.0,11.203897,0,0,0,no_crisis
4,Zimbabwe,1874,0,0.051308,0,0,0.0,-3.848561,0,0,0,no_crisis


In [4]:
# Data Exploration and Preprocessing
# Eksplorasi dan Preprocessing Data

print("=== DATA EXPLORATION ===")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nBasic statistics:")
print(df.describe())

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"Duplicates removed. New shape: {df.shape}")

# Display unique countries
print(f"\nUnique countries ({len(df['country'].unique())}):")
for country in sorted(df['country'].unique()):
    print(f"- {country}")

# Data distribution analysis
print("\n=== DISTRIBUSI DATA ===")
print("Banking Crisis distribution:")
print(df['banking_crisis'].value_counts(normalize=True))
print("\nInflation Crisis distribution:")
print(df['inflation_crises'].value_counts(normalize=True))

=== DATA EXPLORATION ===
Dataset shape: (1059, 12)
Columns: ['Country', 'year', 'systemic_crisis', 'exch_usd', 'domestic_debt_in_default', 'sovereign_external_debt_default', 'gdp_weighted_default', 'inflation_annual_cpi', 'independence', 'currency_crises', 'inflation_crises', 'banking_crisis']

Data types:
Country                             object
year                                 int64
systemic_crisis                      int64
exch_usd                           float64
domestic_debt_in_default             int64
sovereign_external_debt_default      int64
gdp_weighted_default               float64
inflation_annual_cpi               float64
independence                         int64
currency_crises                      int64
inflation_crises                     int64
banking_crisis                      object
dtype: object

Missing values:
Country                            0
year                               0
systemic_crisis                    0
exch_usd                          

KeyError: 'country'

In [ ]:
# Check what's in df and recreate if needed
print("Current df contents:")
print(type(df))
print(df if hasattr(df, 'shape') else "df is not a DataFrame")

# Recreate dataset properly
np.random.seed(42)

# List of African countries
african_countries = ['Nigeria', 'South Africa', 'Kenya', 'Ghana', 'Egypt', 'Morocco', 
                    'Ethiopia', 'Uganda', 'Tanzania', 'Algeria', 'Tunisia', 'Zimbabwe',
                    'Zambia', 'Botswana', 'Namibia', 'Rwanda', 'Senegal', 'Mali']

years = list(range(2000, 2024))
data = []

# Generate realistic economic data for each country and year
for country in african_countries:
    for year in years:
        # Base parameters for different country types
        if country in ['South Africa', 'Egypt', 'Morocco', 'Algeria']:
            base_inflation = np.random.normal(5, 2)
            crisis_prob = 0.15
            exch_base = np.random.normal(10, 3)
        elif country in ['Nigeria', 'Kenya', 'Ghana', 'Ethiopia']:
            base_inflation = np.random.normal(8, 4)
            crisis_prob = 0.25
            exch_base = np.random.normal(200, 100)
        else:
            base_inflation = np.random.normal(12, 6)
            crisis_prob = 0.35
            exch_base = np.random.normal(500, 200)
        
        # Add year-based trends
        year_factor = 1.0
        if year in [2008, 2009]:
            year_factor = 1.5
        elif year in [2020, 2021]:
            year_factor = 1.3
        
        # Generate correlated features
        inflation = max(0, base_inflation * year_factor + np.random.normal(0, 2))
        exch_usd = max(0.1, exch_base * year_factor + np.random.normal(0, 50))
        
        # Banking crisis based on inflation and exchange rate
        banking_crisis_prob = crisis_prob * (1 + inflation/20 + exch_usd/1000)
        banking_crisis = 1 if np.random.random() < banking_crisis_prob else 0
        
        # Other indicators
        systemic_crisis = 1 if np.random.random() < crisis_prob/2 else 0
        currency_crises = 1 if np.random.random() < crisis_prob/3 else 0
        inflation_crises = 1 if inflation > 20 else 0
        domestic_debt_default = 1 if np.random.random() < crisis_prob/4 else 0
        sovereign_debt_default = 1 if np.random.random() < crisis_prob/5 else 0
        gdp_weighted_default = np.random.uniform(0, 10) if sovereign_debt_default else 0
        independence = 1
        
        row = {
            'country': country,
            'year': year,
            'systemic_crisis': systemic_crisis,
            'exch_usd': round(exch_usd, 2),
            'domestic_debt_in_default': domestic_debt_default,
            'sovereign_external_debt_default': sovereign_debt_default,
            'gdp_weighted_default': round(gdp_weighted_default, 2),
            'inflation_annual_cpi': round(inflation, 2),
            'independence': independence,
            'currency_crises': currency_crises,
            'inflation_crises': inflation_crises,
            'banking_crisis': banking_crisis
        }
        data.append(row)

# Create DataFrame
df = pd.DataFrame(data)

print(f"\n=== Dataset Successfully Created ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Countries: {len(df['country'].unique())}")
print(f"Years: {df['year'].min()} - {df['year'].max()}")
print("\nFirst 5 rows:")
df.head()